In [ ]:
import os
os.environ["DEBUG_TWS_CALLBACK"] = "true"
import logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s.%(msecs)03d %(levelname)s: %(message)s', datefmt='%H:%M:%S')
from trading_api.providers.tws.tws_connection import TWSClient
from trading_api.providers.tws import TWSDatafeedProvider
from trading_api.models import QuoteValues
provider = TWSDatafeedProvider()

In [ ]:
results = await provider.search_symbols('AAPL')
results

In [ ]:
data = await provider.get_symbol_info('AAPL', 'SMART')
data

In [ ]:
from datetime import datetime, timedelta
from trading_api.models.market import TimeFrame
import pytz
# Define time range (last week)
end_time = datetime.now().astimezone(pytz.UTC)
start_time = end_time - timedelta(days=7)

# Request hourly bars for AAPL
bars = await provider.get_historical_bars(
    symbol="AAPL",
    start_time=start_time,
    end_time=end_time,
    resolution=TimeFrame.HOUR_1,
    exchange="SMART",  # Optional - uses smart routing
    timeout=30.0,      # Optional - 30 second timeout
)

# Access the bar data
for bar in bars:
    print(f"Time: {bar.time}, Open: {bar.open}, High: {bar.high}, "
          f"Low: {bar.low}, Close: {bar.close}, Volume: {bar.volume}")

In [8]:
quotes = await provider.get_quotes_snapshot(
    symbols=["AAPL"], # , "GOOGL", "MSFT", "TSLA"
    exchange="SMART",  # Optional: smart routing (default)
    timeout=20.0       # Optional: 15s timeout (default)
)

# Process results
for quote in quotes:
    if quote.s == "ok" and isinstance (quote.v, QuoteValues):
        v = quote.v
        print(f"{quote.n}: Last=${v.lp:.2f}, "
                f"Bid=${v.bid:.2f}, Ask=${v.ask:.2f}, "
                f"Volume={v.volume:,}, Change={v.chp:+.2f}%")
    else:
        print(f"{quote.n}: Error - {quote.s}")

16:38:32.349 DEBUG: awaiting tickSnapshotEnd for reqId 12, symbol='AAPL'
16:38:32.560 DEBUG: marketDataType, {'reqId': 12, 'marketDataType': 1}
16:38:32.560 DEBUG: tickReqParams, {'tickerId': 12, 'minTick': 0.01, 'bboExchange': '9c0001', 'snapshotPermissions': 3}
16:38:32.700 DEBUG: tickPrice, {'reqId': 12, 'tickType': 1, 'price': -1.0, 'attrib': 123948481634960: CanAutoExecute: 1, PastLimit: 0, PreOpen: 0}
16:38:32.700 DEBUG: tickSize, {'reqId': 12, 'tickType': 0, 'size': Decimal('0')}
16:38:32.711 DEBUG: tickPrice, {'reqId': 12, 'tickType': 2, 'price': -1.0, 'attrib': 123948475412560: CanAutoExecute: 1, PastLimit: 0, PreOpen: 0}
16:38:32.711 DEBUG: tickSize, {'reqId': 12, 'tickType': 3, 'size': Decimal('0')}
16:38:35.773 DEBUG: tickGeneric, {'reqId': 12, 'tickType': 49, 'value': 0.0}
16:38:35.773 DEBUG: tickSize, {'reqId': 12, 'tickType': 8, 'size': Decimal('0')}
16:38:35.784 DEBUG: tickPrice, {'reqId': 12, 'tickType': 9, 'price': 276.97, 'attrib': 123948475589136: CanAutoExecute: 0,

AAPL: Last=$0.00, Bid=$-1.00, Ask=$-1.00, Volume=0, Change=+0.00%


16:51:34.539 ERROR: Unexpected exception in IBSocket reader loop (running: 0): Socket connection closed.
Traceback (most recent call last):
  File "/home/farouk/trader-pro/backend/src/trading_api/providers/tws/tws_connection.py", line 219, in _reader_loop
    msgId, data, buf, buf_siz = recv(buf, buf_siz)
                                ^^^^^^^^^^^^^^^^^^
  File "/home/farouk/trader-pro/backend/src/trading_api/providers/tws/tws_connection.py", line 376, in receive_data
    assert data, "Socket connection closed."
           ^^^^
AssertionError: Socket connection closed.
16:51:35.040 INFO: IBSocket reader loop finished.


In [ ]:
del provider

In [ ]:
# Test subscribe_realtime_bars and unsubscribe_realtime_bars
import time

# Counter for bars received
bar_count = 0

def on_bar(bar):
    """Callback invoked for each real-time bar."""
    global bar_count
    bar_count += 1
    print(f"[{bar_count}] Real-time bar - Time: {bar.time}, Open: {bar.open}, High: {bar.high}, "
          f"Low: {bar.low}, Close: {bar.close}, Volume: {bar.volume}")

# Subscribe to real-time 5-second bars for AAPL
subscription_id = provider.subscribe_realtime_bars(
    symbol="AAPL",
    callback=on_bar,
    exchange="SMART",
)
print(f"Subscribed with ID: {subscription_id}")

# Let it run for 30 seconds (should receive ~6 bars at 5-second intervals)
print("Listening for real-time bars for 30 seconds...")
time.sleep(20)

# Unsubscribe
provider.unsubscribe_realtime_bars(subscription_id)
print(f"Unsubscribed from real-time bars (received {bar_count} bars)")

In [ ]:
# Test subscribe_market_data and unsubscribe_market_data
import time

# Counter for quotes received
quote_count = 0

def on_quote(quote):
    """Callback invoked for each tick update."""
    global quote_count
    quote_count += 1
    v = quote.v
    print(f"[{quote_count}] {quote.n}: Last=${v.lp:.2f}, Bid=${v.bid:.2f}, Ask=${v.ask:.2f}, Volume={v.volume:,}")

# Subscribe to real-time market data for AAPL
subscription_ids = provider.subscribe_market_data(
    symbols=["AAPL"],
    callback=on_quote,
    exchange="SMART",
)
print(f"Subscribed with IDs: {subscription_ids}")

# Let it run for 20 seconds
print("Listening for real-time market data for 20 seconds...")
time.sleep(20)

# Unsubscribe
provider.unsubscribe_market_data(subscription_ids)
print(f"Unsubscribed from market data (received {quote_count} quotes)")